# NETTOYAGE DES REVIEWS D'ATTRACTIONS TOURISTIQUES DE MARRAKECH


**DESCRIPTION DU DATASET :**

Ce notebook traite le nettoyage des données de reviews (avis) d'attractions touristiques
de Marrakech et sa région. Les données proviennent de TripAdvisor.

`DATASET DES REVIEWS (reviews.csv) :`
- Contient les avis des utilisateurs sur les attractions
- Chaque review inclut : note, texte, date, informations sur le reviewer
- Plus de 50,000 reviews initiales
- Variables clés : rate, review_text, writing_date, writer_informations, stay_info

`DATASET DES ATTRACTIONS (marrakech_attractions_clean.csv) :`
- Dataset déjà nettoyé contenant les informations des attractions
- Inclut : nom, catégorie, note moyenne, nombre d'avis, etc.
- Utilisé pour vérifier la correspondance avec les reviews

`OBJECTIFS DU NOTEBOOK :`
1. Nettoyer et standardiser les données de reviews
2. Corriger les URLs pour assurer la correspondance avec les attractions
3. Extraire des informations structurées (type de voyage, localisation reviewer, etc.)
4. Créer un dataset final prêt pour l'analyse et les systèmes de recommandation

In [102]:
# ============================================================================
# 1. CONFIGURATION ET IMPORTATION
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
import os
import re
import json
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)

# Montage Google Drive
print("🔗 Montage de Google Drive...")
drive.mount('/content/drive')

🔗 Montage de Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [103]:
# ============================================================================
# 2. CHARGEMENT DES DONNÉES
# ============================================================================

print("CHARGEMENT DES FICHIERS...")

# Chemins des fichiers
reviews_path = "/content/drive/MyDrive/Recommender Systems/project/data/reviews.csv"
attractions_path = "/content/drive/MyDrive/Recommender Systems/project/data/marrakech_attractions_clean.csv"

# Chargement avec gestion des erreurs
print(f"Chargement de {reviews_path}...")
reviews_df = pd.read_csv(reviews_path, encoding="latin1", sep=",", on_bad_lines="skip")

print(f"Chargement de {attractions_path}...")
attractions_df = pd.read_csv(attractions_path)

print(f"\nDIMENSIONS INITIALES :")
print(f"Reviews : {reviews_df.shape[0]:,} lignes × {reviews_df.shape[1]} colonnes")
print(f"Attractions : {attractions_df.shape[0]:,} lignes × {attractions_df.shape[1]} colonnes")

print(f"\nAPERÇU DES COLONNES DES REVIEWS :")
for i, col in enumerate(reviews_df.columns, 1):
    print(f"{i:2d}. {col}")

CHARGEMENT DES FICHIERS...
Chargement de /content/drive/MyDrive/Recommender Systems/project/data/reviews.csv...
Chargement de /content/drive/MyDrive/Recommender Systems/project/data/marrakech_attractions_clean.csv...

DIMENSIONS INITIALES :
Reviews : 251,729 lignes × 11 colonnes
Attractions : 4,871 lignes × 12 colonnes

APERÇU DES COLONNES DES REVIEWS :
 1. Unnamed: 0
 2. rate
 3. review_text
 4. review_title
 5. source_url
 6. stay_info
 7. trip_type
 8. writer
 9. writer_informations
10. writer_link
11. writing_date


In [104]:
# ============================================================================
# 3. CORRECTION DES URLS - VERSION CORRIGÉE
# ============================================================================

print("\n" + "="*80)
print("CORRECTION DES URLS - ASSURER LA CORRESPONDANCE AVEC LES ATTRACTIONS")
print("="*80)

"""
PROBLÈME : Les URLs dans le dataset reviews sont des URLs complètes,
tandis que dans le dataset attractions, ce sont des URLs relatives.
Nous devons les standardiser pour pouvoir faire la jointure.

SOLUTION : Extraire la partie relative de l'URL après 'tripadvisor.com'
"""

# Renommer source_url en attraction_url pour plus de clarté
reviews_df = reviews_df.rename(columns={'source_url': 'attraction_url'})

def extract_relative_url_corrected(full_url):
    """Extraire l'URL relative CORRECTEMENT du chemin complet"""
    if pd.isna(full_url):
        return ''

    url_str = str(full_url).strip()

    # Pattern pour extraire le chemin relatif APRÈS tripadvisor.com
    patterns = [
        r'tripadvisor\.com(/Attraction_Review-[^?]+)',  # Le plus probable
        r'(/Attraction_Review-[^?]+)',                  # Fallback
    ]

    for pattern in patterns:
        match = re.search(pattern, url_str)
        if match:
            relative_url = match.group(1)
            # S'assurer qu'il commence par /
            if not relative_url.startswith('/'):
                relative_url = '/' + relative_url
            return relative_url

    # Si aucun pattern ne fonctionne, retourner l'URL telle quelle
    return url_str

def clean_attraction_url(url):
    """Nettoyer les URLs des attractions (dataset existant)"""
    if pd.isna(url):
        return ''

    url_str = str(url).strip()

    # Si l'URL commence par /https://, c'est une erreur, la corriger
    if url_str.startswith('/https://'):
        # Extraire la partie après /https://www.tripadvisor.com
        pattern = r'/https://www\.tripadvisor\.com(/Attraction_Review-[^?]+)'
        match = re.search(pattern, url_str)
        if match:
            return match.group(1)

    return url_str

# Appliquer la correction aux DEUX datasets
print("Nettoyage des URLs dans reviews...")
reviews_df['attraction_url'] = reviews_df['attraction_url'].apply(extract_relative_url_corrected)

print("Nettoyage des URLs dans attractions...")
attractions_df['attraction_url'] = attractions_df['attraction_url'].apply(clean_attraction_url)

# Vérifier quelques exemples
print(f"\nEXEMPLES D'URLS APRÈS CORRECTION :")

# Exemple d'URL d'entrée
test_url = "https://www.tripadvisor.com/Attraction_Review-g1012542-d1117873-Reviews-or10-Berber_Travel_Adventures-Amizmiz_Marrakech_Safi.html"
print(f"URL d'entrée: {test_url}")
print(f"Résultat: {extract_relative_url_corrected(test_url)}")

# Statistiques après correction
print(f"\nSTATISTIQUES APRÈS CORRECTION :")
print(f"URLs uniques dans reviews : {reviews_df['attraction_url'].nunique():,}")
print(f"URLs uniques dans attractions : {attractions_df['attraction_url'].nunique():,}")

# Vérifier la correspondance
common_urls = set(reviews_df['attraction_url']).intersection(set(attractions_df['attraction_url']))
print(f"\nURLs communes : {len(common_urls):,}")
print(f"Couverture : {len(common_urls) / attractions_df['attraction_url'].nunique() * 100:.1f}% des attractions ont des reviews")

if len(common_urls) > 0:
    print("\nExemples d'URLs correspondantes :")
    for i, url in enumerate(list(common_urls)[:3]):
        print(f"{i+1}. {url}")


CORRECTION DES URLS - ASSURER LA CORRESPONDANCE AVEC LES ATTRACTIONS
Nettoyage des URLs dans reviews...
Nettoyage des URLs dans attractions...

EXEMPLES D'URLS APRÈS CORRECTION :
URL d'entrée: https://www.tripadvisor.com/Attraction_Review-g1012542-d1117873-Reviews-or10-Berber_Travel_Adventures-Amizmiz_Marrakech_Safi.html
Résultat: /Attraction_Review-g1012542-d1117873-Reviews-or10-Berber_Travel_Adventures-Amizmiz_Marrakech_Safi.html

STATISTIQUES APRÈS CORRECTION :
URLs uniques dans reviews : 26,130
URLs uniques dans attractions : 4,871

URLs communes : 3,251
Couverture : 66.7% des attractions ont des reviews

Exemples d'URLs correspondantes :
1. /Attraction_Review-g293734-d15224400-Reviews-Marrakech_Personal_Shopper-Marrakech_Marrakech_Safi.html
2. /Attraction_Review-g293734-d5532689-Reviews-I_Go_Morocco-Marrakech_Marrakech_Safi.html
3. /Attraction_Review-g293734-d10622150-Reviews-Marrakech_Travel_Big_Smile-Marrakech_Marrakech_Safi.html


In [105]:
# ============================================================================
# 4. NETTOYAGE DES NOTES
# ============================================================================

print("\n" + "="*80)
print("NETTOYAGE DES NOTES DES REVIEWS")
print("="*80)

"""
Les notes dans le dataset sont au format "4 of 5 bubbles" ou "5 bubbles".
Nous devons extraire la valeur numérique et la normaliser entre 1 et 5.
"""

def clean_review_rate(rate_value):
    """Nettoyer la colonne rate des reviews"""
    if pd.isna(rate_value):
        return np.nan

    try:
        str_value = str(rate_value).strip()
        # Supprimer " of 5 bubbles" et " bubbles"
        str_value = str_value.replace(' of 5 bubbles', '').replace(' bubbles', '')

        # Extraire le nombre
        match = re.search(r'(\d+\.?\d*)', str_value)
        if match:
            value = float(match.group(1))
            # S'assurer que la note est entre 1 et 5
            if 1 <= value <= 5:
                return round(value, 1)  # Arrondir à 1 décimale

        return np.nan
    except Exception as e:
        return np.nan

# Appliquer le nettoyage
reviews_df['rating'] = reviews_df['rate'].apply(clean_review_rate)

print(f"STATISTIQUES DES NOTES :")
print(f"Total reviews : {len(reviews_df):,}")
print(f"Notes valides : {reviews_df['rating'].notna().sum():,} ({reviews_df['rating'].notna().sum()/len(reviews_df)*100:.1f}%)")
print(f"Notes NaN : {reviews_df['rating'].isna().sum():,} ({reviews_df['rating'].isna().sum()/len(reviews_df)*100:.1f}%)")

# Remplacer les notes NaN (utilisateur n'a pas noté) par 0
reviews_df['rating'] = reviews_df['rating'].fillna(0)

print(f"\nREMPLACEMENT DES NOTES MANQUANTES :")
print(f"Total reviews : {len(reviews_df):,}")
print(f"Reviews avec note = 0 (non notées) : {(reviews_df['rating'] == 0).sum():,}")
print(f"Reviews avec notes valides (1 à 5) : {(reviews_df['rating'] > 0).sum():,}")


NETTOYAGE DES NOTES DES REVIEWS
STATISTIQUES DES NOTES :
Total reviews : 251,729
Notes valides : 251,729 (100.0%)
Notes NaN : 0 (0.0%)

REMPLACEMENT DES NOTES MANQUANTES :
Total reviews : 251,729
Reviews avec note = 0 (non notées) : 0
Reviews avec notes valides (1 à 5) : 251,729


In [106]:
# ============================================================================
# 5. NETTOYAGE DES TEXTES
# ============================================================================

print("\n" + "="*80)
print("NETTOYAGE DES TEXTES DES REVIEWS")
print("="*80)

"""
Problèmes à corriger :
1. Encodage incorrect des caractères spéciaux (â€™, â€œ, etc.)
2. Espaces multiples
3. Retours à la ligne et tabulations
"""

def clean_text(text):
    """Nettoyer le texte"""
    if pd.isna(text):
        return ''

    text = str(text)

    # Remplacer les caractères encodés incorrectement
    replacements = {
        'â€™': "'", 'â€œ': '"', 'â€': '"', 'â€"': '-',
        'â€¢': '•', 'â€¦': '...', 'â€˜': "'", '\xa0': ' ',
        '\r': ' ', '\n': ' ', '\t': ' '
    }
    for old, new in replacements.items():
        text = text.replace(old, new)

    # Supprimer les espaces multiples
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# Nettoyer les textes
print("Nettoyage des textes de reviews...")
reviews_df['review_text'] = reviews_df['review_text'].apply(clean_text)
reviews_df['review_title'] = reviews_df['review_title'].apply(clean_text)

# Calculer la longueur
reviews_df['review_length'] = reviews_df['review_text'].str.len()

print(f"\nSTATISTIQUES DES TEXTES :")
print(f"Longueur moyenne : {reviews_df['review_length'].mean():.0f} caractères")
print(f"Reviews avec texte : {(reviews_df['review_length'] > 0).sum():,}/{len(reviews_df):,}")

# SUPPRIMER LES REVIEWS SANS TEXTE OU TROP COURTES
initial_count = len(reviews_df)
reviews_df = reviews_df[reviews_df['review_length'] > 10]
print(f"\nFILTRAGE :")
print(f"Reviews avec texte conservées : {len(reviews_df):,}/{initial_count:,}")
print(f"Reviews supprimées (texte trop court) : {initial_count - len(reviews_df):,}")


NETTOYAGE DES TEXTES DES REVIEWS
Nettoyage des textes de reviews...

STATISTIQUES DES TEXTES :
Longueur moyenne : 314 caractères
Reviews avec texte : 251,701/251,729

FILTRAGE :
Reviews avec texte conservées : 251,243/251,729
Reviews supprimées (texte trop court) : 486


In [107]:
# ============================================================================
# 6. NETTOYAGE DES DATES
# ============================================================================

print("\n" + "="*80)
print("NETTOYAGE DES DATES")
print("="*80)

def parse_review_date(date_str):
    """Parser la date de la review"""
    if pd.isna(date_str):
        return np.nan

    date_str = str(date_str).strip()

    # Supprimer "Written " si présent
    if date_str.startswith('Written '):
        date_str = date_str[8:]

    # Formats à essayer
    formats = [
        '%B %d, %Y',    # January 2, 2023
        '%b %d, %Y',    # Jan 2, 2023
        '%d %B %Y',     # 2 January 2023
        '%Y-%m-%d',     # 2023-01-02
        '%m/%d/%Y',     # 01/02/2023
        '%d/%m/%Y',     # 02/01/2023
    ]

    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt)
        except:
            continue

    # Essayer d'extraire juste l'année
    year_match = re.search(r'(\d{4})', date_str)
    if year_match:
        try:
            return datetime(int(year_match.group(1)), 1, 1)
        except:
            pass

    return np.nan

# Utiliser la colonne writing_date pour les dates
reviews_df['review_date'] = reviews_df['writing_date'].apply(parse_review_date)

# Extraire l'année
reviews_df['review_year'] = reviews_df['review_date'].apply(
    lambda x: x.year if pd.notna(x) else np.nan
)

print(f"Dates extraites : {reviews_df['review_date'].notna().sum()}/{len(reviews_df)}")


NETTOYAGE DES DATES
Dates extraites : 251243/251243


In [108]:
# ============================================================================
# 7. EXTRACTION DES INFORMATIONS AUTEUR
# ============================================================================

print("\n" + "="*80)
print("EXTRACTION DES INFORMATIONS AUTEUR")
print("="*80)

def extract_author_info(info_str):
    """Extraire la localisation et contributions de l'auteur"""
    if pd.isna(info_str):
        return 'Unknown', 0

    info = str(info_str)

    # Localisation
    location = 'Unknown'
    if ',' in info:
        parts = info.split(',')
        for part in parts:
            part = part.strip()
            if part and not re.match(r'^\d+', part) and 'contribution' not in part.lower():
                location = part
                break

    # Contributions
    contributions = 0
    contrib_match = re.search(r'(\d+)\s+contributions?', info)
    if contrib_match:
        contributions = int(contrib_match.group(1))

    return location, contributions

reviews_df['reviewer_location'], reviews_df['reviewer_contributions'] = zip(
    *reviews_df['writer_informations'].apply(extract_author_info)
)

print(f"Auteurs avec localisation : {(reviews_df['reviewer_location'] != 'Unknown').sum()}/{len(reviews_df)}")

# Extraire le nom de l'auteur
def extract_author_name(info_str):
    """Extraire le nom de l'auteur"""
    if pd.isna(info_str):
        return 'Anonymous'

    info = str(info_str)
    if ',' in info:
        first_part = info.split(',')[0].strip()
        if first_part and not re.match(r'^\d+', first_part):
            return first_part
    return 'Anonymous'

reviews_df['reviewer_name'] = reviews_df['writer_informations'].apply(extract_author_name)


EXTRACTION DES INFORMATIONS AUTEUR
Auteurs avec localisation : 132033/251243


In [109]:
# ============================================================================
# 8. EXTRACTION DU TYPE DE VOYAGE
# ============================================================================

print("\n" + "="*80)
print("EXTRACTION DU TYPE DE VOYAGE")
print("="*80)

def extract_trip_type(stay_info):
    """Extraire le type de voyage depuis stay_info"""
    if pd.isna(stay_info):
        return 'Not specified'

    stay_str = str(stay_info)

    # Nettoyer les caractères spéciaux
    stay_str = stay_str.replace('â€¢', '-')

    # Chercher le type après le -
    if '-' in stay_str:
        parts = stay_str.split('-')
        if len(parts) > 1:
            trip_part = parts[1].strip().lower()

            if 'family' in trip_part:
                return 'Family'
            elif 'friends' in trip_part:
                return 'Friends'
            elif 'couple' in trip_part or 'couples' in trip_part:
                return 'Couples'
            elif 'solo' in trip_part:
                return 'Solo'
            elif 'business' in trip_part:
                return 'Business'

    return 'Not specified'

reviews_df['trip_type'] = reviews_df['stay_info'].apply(extract_trip_type)


EXTRACTION DU TYPE DE VOYAGE


In [110]:
# ============================================================================
# 9. EXTRACTION DE L'ANNÉE DE SÉJOUR
# ============================================================================

print("\n" + "="*80)
print("EXTRACTION DE L'ANNÉE DE SÉJOUR")
print("="*80)

def extract_stay_year(stay_info):
    """Extraire l'année de séjour"""
    if pd.isna(stay_info):
        return np.nan

    stay_str = str(stay_info)

    # Chercher une année à 4 chiffres
    year_match = re.search(r'\b(\d{4})\b', stay_str)
    if year_match:
        try:
            return int(year_match.group(1))
        except:
            return np.nan

    return np.nan

reviews_df['stay_year'] = reviews_df['stay_info'].apply(extract_stay_year)


EXTRACTION DE L'ANNÉE DE SÉJOUR


In [111]:
# ============================================================================
# 10. FILTRAGE FINAL
# ============================================================================

print("\n" + "="*80)
print("FILTRAGE FINAL")
print("="*80)

print(f"Reviews avant filtrage : {len(reviews_df)}")

# SUPPRIMER LES REVIEWS POUR DES ATTRACTIONS QUI N'EXISTENT PAS
initial_count = len(reviews_df)
reviews_df = reviews_df[reviews_df['attraction_url'].isin(attractions_df['attraction_url'])]
print(f"Reviews après filtrage (attractions existantes) : {len(reviews_df)}")
print(f"Reviews supprimées (attractions inexistantes) : {initial_count - len(reviews_df)}")

# SUPPRIMER LES DOUBLONS
initial_count = len(reviews_df)
reviews_df = reviews_df.drop_duplicates(
    subset=['review_text', 'reviewer_name', 'attraction_url'],
    keep='first'
)
print(f"Reviews après suppression des doublons : {len(reviews_df)}")
print(f"Doublons supprimés : {initial_count - len(reviews_df)}")


FILTRAGE FINAL
Reviews avant filtrage : 251243
Reviews après filtrage (attractions existantes) : 21875
Reviews supprimées (attractions inexistantes) : 229368
Reviews après suppression des doublons : 21788
Doublons supprimés : 87


In [112]:
# ============================================================================
# 11. CRÉATION DU DATASET FINAL
# ============================================================================

print("\n" + "="*80)
print("CRÉATION DU DATASET FINAL")
print("="*80)

# Sélectionner les colonnes finales
final_columns = [
    'attraction_url',
    'rating',
    'review_title',
    'review_text',
    'review_length',
    'reviewer_name',
    'reviewer_location',
    'reviewer_contributions',
    'trip_type',
    'stay_year',
    'review_year',
    'review_date'
]

reviews_final = reviews_df[final_columns].copy()

# Ajouter un ID unique
reviews_final.insert(0, 'review_id', range(1, len(reviews_final) + 1))

# Formater les dates
if 'review_date' in reviews_final.columns:
    reviews_final['review_date'] = reviews_final['review_date'].apply(
        lambda x: x.strftime('%Y-%m-%d') if pd.notna(x) else ''
    )

print(f"Dimensions finales : {reviews_final.shape}")
print(f"Attractions couvertes : {reviews_final['attraction_url'].nunique()}/{attractions_df.shape[0]}")


CRÉATION DU DATASET FINAL
Dimensions finales : (21788, 13)
Attractions couvertes : 3250/4871


In [113]:
# ============================================================================
# 12. ANALYSE STATISTIQUE
# ============================================================================

print("\n" + "="*80)
print("ANALYSE STATISTIQUE")
print("="*80)

print(f"\nSTATISTIQUES GÉNÉRALES :")
print(f"Nombre total de reviews : {len(reviews_final):,}")
print(f"Note moyenne : {reviews_final['rating'].mean():.2f}")
print(f"Attractions avec reviews : {reviews_final['attraction_url'].nunique():,}")

# Distribution des notes
rating_dist = reviews_final['rating'].value_counts().sort_index()
print(f"\nDISTRIBUTION DES NOTES :")
for rating, count in rating_dist.items():
    percentage = count/len(reviews_final)*100
    print(f"  {rating}: {count:>6} ({percentage:5.1f}%)")


ANALYSE STATISTIQUE

STATISTIQUES GÉNÉRALES :
Nombre total de reviews : 21,788
Note moyenne : 4.77
Attractions avec reviews : 3,250

DISTRIBUTION DES NOTES :
  1.0:    693 (  3.2%)
  2.0:    247 (  1.1%)
  3.0:    321 (  1.5%)
  4.0:    863 (  4.0%)
  5.0:  19664 ( 90.3%)


In [114]:
# ============================================================================
# 13. SAUVEGARDE
# ============================================================================

print("\n" + "="*80)
print("SAUVEGARDE")
print("="*80)

# Sauvegarder le dataset final
output_path = "/content/drive/MyDrive/Recommender Systems/project/data/marrakech_reviews_clean.csv"
reviews_final.to_csv(output_path, index=False, encoding='utf-8')

print(f"Dataset sauvegardé : {output_path}")
print(f"Taille : {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

# Aperçu final
print(f"\nAPERÇU FINAL :")
print(f"{len(reviews_final):,} reviews nettoyées")
print(f"{reviews_final['attraction_url'].nunique():,} attractions couvertes")

print("\n" + "="*80)
print("NETTOYAGE TERMINÉ AVEC SUCCÈS!")
print("="*80)


SAUVEGARDE
Dataset sauvegardé : /content/drive/MyDrive/Recommender Systems/project/data/marrakech_reviews_clean.csv
Taille : 10.27 MB

APERÇU FINAL :
21,788 reviews nettoyées
3,250 attractions couvertes

NETTOYAGE TERMINÉ AVEC SUCCÈS!
